# 2. Limpeza dos Dados (versão explicada passo a passo)

> Este notebook é uma cópia **funcional** de `notebooks/02_data_cleaning.ipynb`, com o mesmo código e os mesmos resultados, mas com explicações detalhadas de cada linha. Os conceitos básicos (o que é DataFrame, Series, CSV, f-string, list comprehension) estão explicados na seção 0 e no glossário de `01_data_inspection_explicado.ipynb`; aqui eles são usados sem repetir a explicação.

**O que esta etapa faz.** A inspeção (notebook 01) só **observou** os dados e anotou os problemas. A limpeza é a etapa que **corrige** esses problemas e produz uma base pronta para a análise exploratória e para a modelagem. A lista de tratamentos vem direto da seção 1.11 da inspeção:

| Item | Situação encontrada na inspeção | Tratamento |
|---|---|---|
| `Unnamed: 32` | 100% vazia, gerada pela vírgula ao final do cabeçalho | Remover |
| `id` | Identificador único, sem valor preditivo | Remover |
| `concave points_*` | Espaço no nome da coluna | Renomear para `concave_points_*` |
| `diagnosis` | Categórica (`M`/`B`) | Converter para `1`/`0` |
| Valores ausentes | Nenhum nas variáveis de medida | Verificar e confirmar |
| Duplicatas | Nenhuma | Verificar e confirmar |
| Zeros em concavidade | 13 exames, valores plausíveis | Manter |
| Outliers (IQR) | 171 exames (30,1%), 2/3 deles malignos | Manter |

**Regra de ouro:** o arquivo bruto em `data/raw/` **nunca é modificado**. A limpeza lê o bruto, trabalha em uma cópia na memória e grava o resultado em um arquivo novo, `data/processed/breast_cancer_wisconsin_clean.csv`. Assim, se alguém discordar de um tratamento, basta mudar o notebook e rodar de novo; o original continua intacto.


## 2.1 Importação das bibliotecas

**O que vamos fazer:** carregar o que precisamos. A limpeza não desenha gráficos, então não importamos Matplotlib nem Seaborn.

**Linha por linha:**

- `from pathlib import Path`: importa a classe `Path`, da biblioteca padrão do Python, que representa caminhos de arquivos e pastas. Vamos usá-la na seção 2.10 para criar a pasta `data/processed/` caso ela não exista. A forma `from X import Y` traz só o item `Y` de dentro do módulo `X`, em vez do módulo inteiro.
- `import pandas as pd`: o pandas, como sempre.
- `pd.set_option("display.max_columns", 12)`: limita a exibição a 12 colunas, como no notebook 01.


In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 12)


## 2.2 Carregamento da base

**O que vamos fazer:** ler o CSV bruto e guardar **duas** versões dele: uma que será modificada e uma que fica intacta para comparação.

**Linha por linha:**

- `DATA_PATH = "../data/raw/breast_cancer_wisconsin.csv"`: caminho do arquivo bruto, relativo à pasta deste notebook (`..` sobe para a raiz do projeto).
- `OUTPUT_PATH = "../data/processed/breast_cancer_wisconsin_clean.csv"`: caminho onde a base limpa será gravada. Definir os dois caminhos juntos, no topo, facilita achar e mudar depois.
- `df_raw = pd.read_csv(DATA_PATH)`: lê o CSV e guarda em `df_raw` (*raw* = bruto).
- `df = df_raw.copy()`: cria uma **cópia independente** e guarda em `df`. Todos os tratamentos serão feitos em `df`; `df_raw` fica como estava. O `.copy()` é importante: se escrevêssemos só `df = df_raw`, as duas variáveis apontariam para **a mesma tabela** na memória, e mudar uma mudaria a outra.
- Os dois `print` mostram as dimensões iniciais, para comparar com o final.


In [2]:
DATA_PATH = "../data/raw/breast_cancer_wisconsin.csv"
OUTPUT_PATH = "../data/processed/breast_cancer_wisconsin_clean.csv"

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

print(f"Linhas: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")


Linhas: 569
Colunas: 33


**Como ler o resultado:** 569 linhas e 33 colunas, exatamente como na inspeção. A partir daqui, a contagem de colunas vai diminuindo a cada tratamento.


## 2.3 Remoção da coluna vazia `Unnamed: 32`

**O que vamos fazer:** apagar a coluna fantasma que o pandas criou por causa da vírgula sobrando no cabeçalho do CSV.

Em vez de escrever o nome `Unnamed: 32` diretamente, o código **procura** qualquer coluna 100% vazia. O efeito é o mesmo, mas o notebook continua correto se o CSV for corrigido no futuro (sem a vírgula final, não haveria coluna vazia e a lista ficaria simplesmente vazia, sem erro).

**Linha por linha:**

- `df.isnull()`: tabela de `True`/`False` com `True` onde o valor é NaN.
- `.all()`: para cada **coluna**, responde "todos os valores são `True`?". Resultado: uma Series com um booleano por coluna, `True` só nas colunas inteiramente vazias.
- `df.columns[...]`: usa essa Series como filtro sobre os nomes das colunas, ficando só com os nomes das vazias. `.tolist()` converte em lista.
- `df = df.drop(columns=empty_cols)`: `drop` devolve uma tabela **sem** essas colunas. Como queremos que a mudança valha daqui para frente, guardamos o resultado de volta em `df` (na inspeção, o `drop` era usado sem atribuição justamente para **não** alterar `df`).


In [3]:
empty_cols = df.columns[df.isnull().all()].tolist()
print(f"Colunas totalmente vazias: {empty_cols}")

df = df.drop(columns=empty_cols)
print(f"Colunas após a remoção: {df.shape[1]}")


Colunas totalmente vazias: ['Unnamed: 32']
Colunas após a remoção: 32


**Como ler o resultado:** apenas `Unnamed: 32` atende ao critério. Depois da remoção, sobram 32 colunas.


## 2.4 Remoção da coluna `id`

**O que vamos fazer:** apagar o identificador do exame.

**Por quê?** O `id` é um número de cadastro. Ele não diz nada sobre o tumor, então não pode ajudar a prever o diagnóstico. Pior: se ficasse na base, um modelo poderia "decorar" que o exame 842302 é maligno, o que parece um acerto no treino mas não serve para nenhum exame novo. Isso se chama **vazamento** de informação e é um erro clássico.

**Linha por linha:**

- `df['id'].nunique()`: conta quantos valores **distintos** existem na coluna (*number of unique*). Comparar com `len(df)` (o número de linhas) confirma que cada exame tem um id próprio; se houvesse ids repetidos, valeria investigar antes de descartar a coluna.
- `df = df.drop(columns=["id"])`: remove a coluna e guarda o resultado em `df`.


In [4]:
print(f"IDs únicos: {df['id'].nunique()} de {len(df)} registros")

df = df.drop(columns=["id"])
print(f"Colunas após a remoção: {df.shape[1]}")


IDs únicos: 569 de 569 registros
Colunas após a remoção: 31


**Como ler o resultado:** 569 ids únicos para 569 registros, então a coluna pode ser descartada sem perda de informação. Sobram 31 colunas: `diagnosis` e as 30 medidas.


## 2.5 Padronização dos nomes das colunas

**O que vamos fazer:** trocar o espaço por `_` nos nomes `concave points_mean`, `concave points_se` e `concave points_worst`.

**Por quê?** Todas as outras colunas seguem o padrão `caracteristica_sufixo`. Nomes com espaço obrigam a escrever `df["concave points_mean"]` (com aspas e colchetes) e não funcionam com a sintaxe curta `df.concave_points_mean`; algumas bibliotecas de modelagem também reclamam. É um ajuste cosmético, mas evita dor de cabeça.

**Linha por linha:**

- `{col: col.replace(" ", "_") for col in df.columns if " " in col}`: uma **dict comprehension**, prima da list comprehension, que monta um **dicionário** em vez de uma lista. Para cada coluna que contém espaço, cria um par `nome antigo: nome novo`, em que o nome novo é o antigo com os espaços trocados por `_` (`replace` = substituir).
- `df = df.rename(columns=renamed)`: `rename` recebe o dicionário e troca os nomes. Colunas que não estão no dicionário ficam como estão.
- `renamed` sozinho na última linha exibe o dicionário, para conferir o que foi trocado.


In [5]:
renamed = {col: col.replace(" ", "_") for col in df.columns if " " in col}
df = df.rename(columns=renamed)

renamed


{'concave points_mean': 'concave_points_mean',
 'concave points_se': 'concave_points_se',
 'concave points_worst': 'concave_points_worst'}

**Como ler o resultado:** três pares `antigo: novo`, todos de `concave points`. Nenhuma outra coluna tinha espaço.


## 2.6 Conversão da variável alvo `diagnosis`

**O que vamos fazer:** transformar o texto `M`/`B` nos números `1`/`0`.

**Por quê?** Os algoritmos de machine learning trabalham com números. Escolhemos `1` para **maligno** porque ele é a **classe positiva**: o caso que o modelo precisa detectar. Essa escolha não é só convenção; ela define o que as métricas medem. O **recall** é calculado sobre a classe `1`, então com `1` = maligno ele responde exatamente à pergunta que interessa: "de todos os tumores malignos, quantos o modelo pegou?". Se tivéssemos usado `1` = benigno, o recall mediria a coisa errada.

**Linha por linha:**

- `sorted(df['diagnosis'].unique())`: `unique` lista os valores distintos da coluna; `sorted` ordena. Serve para confirmar que só existem `B` e `M` (nenhum `m` minúsculo, `Maligno` por extenso ou espaço sobrando). Se houvesse uma variante, o `map` da linha seguinte a transformaria em NaN sem avisar.
- `df["diagnosis"].map({"M": 1, "B": 0})`: `map` com um dicionário troca cada valor pelo correspondente. Igual ao que fizemos na seção 1.10 da inspeção, mas agora o resultado é **gravado de volta** na coluna (`df["diagnosis"] = ...`).
- `.astype("int64")`: força o tipo da coluna para inteiro. O `map` já devolve inteiros, mas se algum valor tivesse virado NaN a coluna seria `float64`; o `astype` daria erro nesse caso, o que é bom: é um alarme.
- O `pd.DataFrame({...})` final monta a contagem e o percentual por classe, como na seção 1.9 da inspeção, para comparar.


In [6]:
print(f"Valores encontrados: {sorted(df['diagnosis'].unique())}")

df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0}).astype("int64")

pd.DataFrame({
    "quantidade": df["diagnosis"].value_counts(),
    "percentual": (df["diagnosis"].value_counts(normalize=True) * 100).round(2),
})


Valores encontrados: ['B', 'M']


,quantidade,percentual
diagnosis,,
0,357,62.74
1,212,37.26


**Como ler o resultado:** só `B` e `M` existiam. Depois da conversão, a tabela mostra `0` com 357 exames (62,74%) e `1` com 212 (37,26%), os mesmos números da inspeção. Se algum valor tivesse ficado sem mapeamento, a soma seria menor que 569.


## 2.7 Valores ausentes e registros duplicados

**O que vamos fazer:** confirmar que, depois de tirar a coluna fantasma, não sobrou nenhum NaN, e que não há linhas repetidas.

A inspeção já tinha mostrado que não havia nenhum dos dois. Mesmo assim a verificação é repetida aqui, porque a limpeza deve ser **autossuficiente**: quem rodar só este notebook precisa ter certeza de que a base está completa.

**Linha por linha:**

- `df.isnull().sum().sum()`: total de NaN na tabela inteira (o primeiro `sum` conta por coluna; o segundo soma as colunas).
- `df.duplicated().sum()`: quantas linhas são cópias exatas de uma linha anterior.
- `rows_before = len(df)`: guarda o número de linhas antes.
- `df = df.drop_duplicates()`: remove as linhas repetidas (mantendo a primeira ocorrência). Aqui não remove nada, mas a chamada fica por segurança.
- `rows_before - len(df)`: quantas linhas saíram.


In [7]:
print(f"Valores ausentes: {df.isnull().sum().sum()}")
print(f"Registros duplicados: {df.duplicated().sum()}")

rows_before = len(df)
df = df.drop_duplicates()
print(f"Registros removidos por duplicidade: {rows_before - len(df)}")


Valores ausentes: 0
Registros duplicados: 0
Registros removidos por duplicidade: 0


**Como ler o resultado:** zero ausentes, zero duplicados, zero linhas removidas. Nenhuma estratégia de **imputação** (preencher valores faltantes com média, mediana etc.) é necessária. Se a base tivesse alguns NaN, o caminho usual seria preencher com a mediana da coluna; se uma coluna tivesse muitos, seria melhor removê-la.


## 2.8 Zeros e outliers: decisão de manter

**O que vamos fazer:** recontar os zeros nas colunas de concavidade e os outliers pelo critério IQR, para registrar no próprio notebook de limpeza a decisão de **não mexer** neles.

Uma limpeza não é só apagar; é também decidir, com justificativa, o que fica. O código é o mesmo da seção 1.6 da inspeção, com dois ajustes: os nomes das colunas agora usam `concave_points` (renomeadas na 2.5) e o diagnóstico é `1`/`0`.

**Linha por linha:**

- `col.startswith(("concavity", "concave_points"))`: `startswith` aceita uma **tupla** de prefixos e devolve `True` se o nome começa com qualquer um deles. A list comprehension monta a lista das seis colunas de concavidade.
- `(df[concavity_cols] == 0).any(axis=1)`: para cada linha, "há zero em alguma dessas seis colunas?". `.sum()` conta os exames.
- `features = df.drop(columns=["diagnosis"])`: só as 30 medidas, sem o alvo (que agora é numérico e entraria na conta por engano).
- As linhas de `q1`, `q3`, `iqr`, `is_outlier` e `rows_with_outlier` são idênticas às da inspeção: quartis, intervalo interquartil e marcação dos valores fora de Q1 − 1,5·IQR e Q3 + 1,5·IQR.
- `df.loc[rows_with_outlier, "diagnosis"].value_counts()`: quantos exames com outlier são malignos (`1`) e quantos benignos (`0`).


In [8]:
concavity_cols = [col for col in df.columns if col.startswith(("concavity", "concave_points"))]
zero_rows = (df[concavity_cols] == 0).any(axis=1)
print(f"Exames com zero nas colunas de concavidade: {zero_rows.sum()}")

features = df.drop(columns=["diagnosis"])
q1, q3 = features.quantile(0.25), features.quantile(0.75)
iqr = q3 - q1
is_outlier = (features < q1 - 1.5 * iqr) | (features > q3 + 1.5 * iqr)
rows_with_outlier = is_outlier.any(axis=1)

print(f"Exames com ao menos um outlier (IQR): {rows_with_outlier.sum()} ({rows_with_outlier.mean() * 100:.1f}%)")
print("Outliers por classe (1 = maligno):")
print(df.loc[rows_with_outlier, "diagnosis"].value_counts().to_string())


Exames com zero nas colunas de concavidade: 13
Exames com ao menos um outlier (IQR): 171 (30.1%)
Outliers por classe (1 = maligno):
diagnosis
1    114
0     57


**Como ler o resultado:** 13 exames com zero nas colunas de concavidade e 171 exames (30,1%) com ao menos um outlier, sendo 114 malignos e 57 benignos. Os mesmos números da inspeção, o que confirma que nenhum tratamento anterior alterou valores.

**Por que manter os zeros?** Um núcleo com contorno perfeitamente liso tem concavidade zero. É um valor **real**, não um "não sei" disfarçado. Tratar como ausente e preencher com a mediana inventaria uma concavidade que a célula não tem.

**Por que manter os outliers?** Dois terços deles são malignos, numa base em que só 37% dos exames são malignos. Ou seja, os valores extremos são justamente os tumores mais agressivos, com núcleos maiores e mais irregulares. Removê-los seria jogar fora os casos que o modelo mais precisa reconhecer. As diferenças de escala e a assimetria serão tratadas no pré-processamento, com padronização, transformação logarítmica ou escalonador robusto, dependendo do modelo.


## 2.9 Validação da base limpa

**O que vamos fazer:** conferir, de forma automática, que a base ficou exatamente como esperado.

### O que é `assert`?
`assert condicao` é uma instrução do Python que **verifica** uma condição: se ela for verdadeira, nada acontece e o código segue; se for falsa, o notebook **para com um erro** (`AssertionError`). É uma forma de escrever "tenho certeza de que isso é verdade; se não for, me avise na hora". Se alguém mudar um tratamento lá em cima e quebrar alguma garantia, esta célula acusa.

**Linha por linha:**

- `assert df.shape == (569, 31), df.shape`: 569 linhas e 31 colunas. O texto depois da vírgula (`df.shape`) é exibido junto com o erro, caso falhe, para mostrar qual foi o formato encontrado.
- `assert df.isnull().sum().sum() == 0`: nenhum NaN.
- `assert df.duplicated().sum() == 0`: nenhuma linha duplicada.
- `assert set(df["diagnosis"].unique()) == {0, 1}`: o alvo só tem 0 e 1. `set` transforma a lista em um **conjunto** (coleção sem ordem e sem repetição), para a comparação não depender da ordem.
- `assert df.drop(columns=["diagnosis"]).dtypes.eq("float64").all()`: todas as 30 medidas são `float64`. `dtypes` lista os tipos; `.eq("float64")` compara cada um com `float64`; `.all()` exige que todos sejam `True`.
- `assert not any(" " in col for col in df.columns)`: nenhum nome de coluna contém espaço. `any` devolve `True` se ao menos um for `True`; o `not` inverte.
- Se todos passarem, o `print` confirma e o `df.info()` mostra o resumo final.


In [9]:
assert df.shape == (569, 31), df.shape
assert df.isnull().sum().sum() == 0
assert df.duplicated().sum() == 0
assert set(df["diagnosis"].unique()) == {0, 1}
assert df.drop(columns=["diagnosis"]).dtypes.eq("float64").all()
assert not any(" " in col for col in df.columns)

print("Validação concluída sem erros.")
df.info()


Validação concluída sem erros.
<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   diagnosis                569 non-null    int64  
 1   radius_mean              569 non-null    float64
 2   texture_mean             569 non-null    float64
 3   perimeter_mean           569 non-null    float64
 4   area_mean                569 non-null    float64
 5   smoothness_mean          569 non-null    float64
 6   compactness_mean         569 non-null    float64
 7   concavity_mean           569 non-null    float64
 8   concave_points_mean      569 non-null    float64
 9   symmetry_mean            569 non-null    float64
 10  fractal_dimension_mean   569 non-null    float64
 11  radius_se                569 non-null    float64
 12  texture_se               569 non-null    float64
 13  perimeter_se             569 non-null    float64
 14  area_s

In [10]:
df.head()


,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,...,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst
0,1,17.99,10.38,122.80,1001.0,0.11840,...,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,1,20.57,17.77,132.90,1326.0,0.08474,...,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,1,19.69,21.25,130.00,1203.0,0.10960,...,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,1,11.42,20.38,77.58,386.1,0.14250,...,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,1,20.29,14.34,135.10,1297.0,0.10030,...,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


**Como ler o resultado:** o `info()` mostra 31 colunas, todas com `569 non-null`; `diagnosis` como `int64` e as demais como `float64`. O `head()` mostra as cinco primeiras linhas: sem `id`, sem `Unnamed: 32`, com `diagnosis` igual a `1` (os cinco primeiros exames são malignos) e com `concave_points_mean` no lugar de `concave points_mean`.


## 2.10 Salvamento da base tratada

**O que vamos fazer:** gravar `df` em um CSV novo e conferir que o arquivo gravado é idêntico ao que está na memória.

**Linha por linha:**

- `Path(OUTPUT_PATH).parent`: `Path` transforma o texto do caminho em um objeto que entende pastas; `.parent` é a pasta que contém o arquivo (`../data/processed`).
- `.mkdir(parents=True, exist_ok=True)`: cria essa pasta. `parents=True` cria também as pastas intermediárias, se faltarem; `exist_ok=True` não dá erro se a pasta já existir. Sem isso, o `to_csv` falharia na primeira execução, quando `data/processed/` ainda não existe.
- `df.to_csv(OUTPUT_PATH, index=False)`: grava o DataFrame como CSV. `index=False` evita gravar o índice (`0, 1, 2, ...`) como uma coluna extra, que apareceria como `Unnamed: 0` na próxima leitura. Seria criar um problema igual ao que acabamos de resolver.
- `df_saved = pd.read_csv(OUTPUT_PATH)`: relê o arquivo recém-gravado.
- `pd.testing.assert_frame_equal(df_saved, df)`: compara as duas tabelas célula a célula, incluindo nomes e tipos das colunas. Se houver qualquer diferença (por exemplo, um número que perdeu precisão ao ser gravado como texto), dá erro. Se passar, temos certeza de que o arquivo em disco é fiel ao que validamos.


In [11]:
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

df_saved = pd.read_csv(OUTPUT_PATH)
pd.testing.assert_frame_equal(df_saved, df)
print(f"Base salva em {OUTPUT_PATH}: {df_saved.shape[0]} linhas x {df_saved.shape[1]} colunas")


Base salva em ../data/processed/breast_cancer_wisconsin_clean.csv: 569 linhas x 31 colunas


**Como ler o resultado:** a mensagem confirma o caminho e as dimensões (569 × 31). Se a comparação tivesse falhado, o erro apareceria antes do `print`.


## 2.11 Conclusões da limpeza

Para fechar, uma tabela comparando a base bruta (`df_raw`, intocada desde a leitura) com a base limpa (`df`).

**Linha por linha:** `pd.DataFrame` recebe um dicionário com duas colunas (`bruta` e `limpa`), cada uma com quatro valores na mesma ordem: número de linhas, número de colunas, total de NaN e o tipo da coluna `diagnosis` (convertido para texto com `str` para caber na tabela). O `index=[...]` dá nome a cada linha.


In [12]:
pd.DataFrame({
    "bruta": [df_raw.shape[0], df_raw.shape[1], df_raw.isnull().sum().sum(), str(df_raw["diagnosis"].dtype)],
    "limpa": [df.shape[0], df.shape[1], df.isnull().sum().sum(), str(df["diagnosis"].dtype)],
}, index=["linhas", "colunas", "valores ausentes", "tipo de diagnosis"])


,bruta,limpa
linhas,569,569
colunas,33,31
valores ausentes,569,0
tipo de diagnosis,str,int64


**Como ler o resultado:** as linhas continuam 569; as colunas caíram de 33 para 31; os 569 valores ausentes (todos da coluna fantasma) viraram 0; e `diagnosis` passou de texto (`str`) para inteiro (`int64`).

| Tratamento | Resultado | Por quê |
|---|---|---|
| Remoção de `Unnamed: 32` | 1 coluna removida | Coluna sem nenhuma informação, só NaN |
| Remoção de `id` | 1 coluna removida | Identificador sem relação com o tumor; evitaria "decorar" exames |
| Renomeação de `concave points_*` | 3 colunas renomeadas | Padronizar nomes e facilitar o acesso no código |
| Conversão de `diagnosis` | `M` → `1`, `B` → `0` | Modelos trabalham com números; `1` = classe de interesse |
| Valores ausentes | Nenhum encontrado, nenhuma imputação | |
| Duplicatas | Nenhuma encontrada, nenhuma linha removida | |
| Zeros e outliers | Mantidos | São valores reais e casos clinicamente relevantes |

A base passou de 33 para 31 colunas e manteve os 569 exames. **Nenhum valor foi alterado ou imputado**; as únicas mudanças são estruturais (remoção de colunas, renomeação e codificação do alvo). A base tratada em `data/processed/breast_cancer_wisconsin_clean.csv` é o ponto de partida da análise exploratória (`03_eda.ipynb`) e da modelagem.

**Pendências para as etapas seguintes** (não são tarefas da limpeza, mas dependem do que foi visto aqui e na inspeção)

- **Padronização** das variáveis, dada a diferença de escala (área na casa dos milhares, suavidade abaixo de 1).
- Avaliar **transformação logarítmica** ou **escalonador robusto** para as variáveis `_se`, muito assimétricas.
- Considerar **seleção de atributos** por causa da multicolinearidade entre raio, perímetro e área e entre as versões `_mean` e `_worst`.
- Usar **separação estratificada** entre treino e teste por causa do desbalanceamento moderado das classes.


## Glossário

Os termos já definidos no glossário de `01_data_inspection_explicado.ipynb` (DataFrame, Series, NaN, IQR, outlier, padronização, recall, separação estratificada etc.) não são repetidos aqui.

| Termo | Significado |
|---|---|
| **`assert`** | Instrução que verifica uma condição e interrompe o código com erro se ela for falsa. Usada para garantir que a base ficou como esperado. |
| **Classe positiva** | A classe que o modelo deve detectar, codificada como `1`. Aqui, maligno. Métricas como recall e precisão são calculadas sobre ela. |
| **Conjunto (`set`)** | Coleção do Python sem ordem e sem repetição. Útil para comparar "quais valores existem" sem se importar com a ordem. |
| **Cópia (`copy`)** | Duplicata independente de uma tabela. Sem ela, duas variáveis podem apontar para a mesma tabela e uma mudança em uma afeta a outra. |
| **Dicionário** | Estrutura do Python com pares `chave: valor`, escrita entre chaves `{}`. Usado em `map` e `rename` para dizer "troque isto por aquilo". |
| **Dict comprehension** | Forma compacta de criar um dicionário: `{chave: valor for x in lista if condicao}`. |
| **Imputação** | Preencher valores ausentes com um valor estimado (média, mediana, moda). Não foi necessária nesta base. |
| **`map`** | Função que troca cada valor de uma coluna pelo correspondente em um dicionário. |
| **`Path`** | Classe da biblioteca padrão do Python para lidar com caminhos de arquivos e pastas. |
| **Vazamento (leakage)** | Quando uma informação que não estaria disponível na hora da previsão real entra no treino do modelo, inflando a avaliação. Manter o `id` seria um exemplo. |
